# Menos odio a la vista: detección de comentarios agresivos en español
**Proyecto de Aplicación 2: Hugging Face** · Técnicas de Aprendizaje de Máquina · Pontificia Universidad Javeriana

*Un filtro previo para cuidar a quien lee —comunidades, chicas, niños y adolescentes— sin apagar la conversación.*

Integrantes: _(completar)_

> Estado: borrador técnico con resultados verificados (seed 42, partición por artículo). Las cifras de modelado se obtuvieron en máquina local (Apple Silicon MPS / CPU) y deben reproducirse en Colab al ejecutar las celdas de la sección 5.


# 0. Elevator pitch

Internet dejó de ser un rincón para adultos: es el patio, el recreo y a veces el único espacio donde una chica, un niño o cualquier persona se encuentra con el mundo. En ese mismo espacio, un comentario de odio puede llegar más rápido que un abrazo.

**Pregunta que nos mueve:** ¿podemos detectar a tiempo, en español, un comentario que agrede o incita al odio, y reducir la exposición de quien lee —sobre todo de comunidades vulnerables y menores— sin esperar a que un moderador humano llegue “cuando pueda”?

**Por qué importa (para quien podría financiar esto):**
1. **Cuidado de las personas:** el boom de redes, blogs y foros multiplica comentarios que normalizan discriminación, burla y violencia verbal. No afirmamos que un modelo “cure” la salud mental; sí que **menos exposición a odio** es un gesto concreto de protección hacia quien navega, incluyendo niñas, niños y adolescentes.
2. **Comunidades más humanas:** detrás de cada hilo hay gente real. Un filtro previo con aviso de normas permite conversar sin que el primer mensaje que vea alguien sea un ataque.
3. **Moderación al alcance de equipos pequeños:** diarios, blogs y foros en español rara vez tienen personal 24/7. Automatizar la primera línea (ocultar / revisar) libera a las personas para decidir los casos difíciles, no para leer insultos todo el día.

**Cómo se entrega:** (1) modelo de clasificación de texto en español, (2) aplicación web desplegada que simula un blog: publicás un comentario agresivo y el sistema lo oculta con el mensaje de normas de la comunidad, (3) demo extra del equipo: extensión de navegador que aplica el mismo criterio fuera del blog.

**Justificación del enfoque y síntesis de la revisión crítica:** una lista de groserías solo alcanza F1≈0,20 en DEV: el odio viaja sin insultos y hay insultos sin odio. Comparar tres arquitecturas + línea base sin ajuste muestra que adaptar RoBERTuito al corpus (A3) supera a MLP, BiLSTM y al clasificador público multietiqueta. La revisión con IA (sección 9) empujó a declarar límites (no es terapia; riesgo de censurar contra-discurso) y a dejar humano en el loop con aviso de normas.


# 1. Fase 0: definición del problema

## 1.1 Problema (técnico y social)
**Social:** en un internet saturado de comentarios, el odio ya no es un "ruido" aislado: puede ser lo primero que lee alguien que solo quería informarse, participar o sentirse parte de una comunidad. Nos preocupa especialmente quiénes tienen menos herramientas para filtrar ese impacto —chicas, niños, adolescentes y personas de grupos históricamente agredidos—. El proyecto no promete "salvar" la salud mental; busca **interrumpir a tiempo** la llegada de mensajes que agreden, para que el espacio digital no se sienta hostil por defecto.

**Técnico:** clasificación **binaria** de un comentario en español: *con odio* (`HATEFUL = 1`) o *sin odio* (`HATEFUL = 0`). Extensión opcional: multietiqueta por tipo de odio (mujeres, LGBTI, racismo, clase, política, discapacidad, apariencia, criminales) y detección de llamados a la acción (`CALLS`).

## 1.2 Entrada y salida
- **Unidad de análisis:** un comentario (tuit de respuesta) como texto. El corpus incluye además el contexto (título y texto de la noticia a la que responde); se evaluará si ese contexto mejora el modelo.
- **Entrada:** texto del comentario (hasta 307 caracteres en el corpus; la app debe informar el límite y qué pasa con textos más largos).
- **Salida:** probabilidad de odio en [0, 1] y clase según un umbral configurable. En la app, esa salida se traduce a una acción humana: ocultar con aviso de normas de la comunidad (y opción de mostrar / revisar), no a un borrado silencioso.

## 1.3 Datos y referencias de evaluación
- **Corpus principal:** `piuba-bigdata/contextualized_hate_speech`. 56.869 filas (train 36.420, dev 9.106, test 11.343). Tuits argentinos en respuesta a noticias de Clarín, Infobae, La Nación, Perfil y Crónica durante la pandemia de COVID-19.
- **Etiquetas:** anotadas por personas. `HATEFUL` y `CALLS` se binarizan por mayoría simple; las categorías se marcan en 1 si al menos un anotador las marcó. Existe una versión sin agregar (`piuba-bigdata/contextualized_hate_speech_raw`) que permitiría medir el desacuerdo entre anotadores. _(TODO: leer el artículo arXiv 2210.00465 y registrar el acuerdo entre anotadores.)_
- **Fuentes complementarias:** `Paul/hatecheck-spanish` (casos de prueba funcionales para el análisis de errores) y una muestra fuera de dominio (por ejemplo `SINAI/ALIA-es-discriminative-hate-speech`, comentarios de YouTube/TikTok con etiquetas generadas por LLM, tratadas como referencia débil). _(TODO: confirmar acceso y licencias.)_
- **Limitación a declarar:** dialecto argentino y tema pandémico; la app podría fallar con otros países y otros temas. El cuidado empático del pitch no borra este límite técnico.

## 1.4 Propuesta de valor
**Para quién:** administradores de blogs, foros y medios en español que quieren cuidar a su audiencia —incluidas familias y menores que entran a leer— sin tener un equipo de moderación permanente.

**Cómo se usa:** filtro previo con empatía hacia dos lados: protege a quien lee (menos exposición a odio) y deja espacio a la conversación legítima (aviso visible, "mostrar de todos modos", escalado a una persona). **No sustituye** el juicio humano ni impone censura absoluta: reduce daño evitable mientras se decide.

## 1.5 Justificación del enfoque
- **¿Por qué no una regla o lista de palabras?** El odio a menudo llega disfrazado de "humor", sarcasmo o frases sin groserías; y a veces hay groserías sin intención de odio. Una lista rígida castiga de más o de menos a personas reales. _(TODO: sustentar con conteo real de la sección 3.)_
- **¿Por qué comparar si ya existe un modelo preentrenado?** Los modelos multilingües de toxicidad se entrenaron con otras definiciones y dominios. La línea base mide cuánto aporta adaptar un modelo al español rioplatense y a la definición de "odio" de este corpus —para no desplegar un filtro que falle justo donde más duele.
- **Hipótesis a comprobar:** (H1) un Transformer ajustado supera a MLP y BiLSTM; (H2) la línea base sin ajuste tiene recall alto pero precisión baja por diferencia de definición; (H3) agregar el contexto de la noticia mejora los resultados.

## 1.6 Revisión crítica con IA de la Fase 0
Registrar en la sección 9 el prompt, la evidencia que recibió la IA, la crítica y la decisión del equipo. Prompts sugeridos (ajustados al enfoque social):
> "Actúa como madre, padre o cuidador de un adolescente que lee blogs y redes en español. Queremos detectar comentarios con odio para ocultarlos con un aviso de normas. ¿Qué cinco razones tendrías para no confiar en esta solución y qué evidencia te haría cambiar de opinión?"

> "Actúa como administrador de una comunidad en línea preocupado por la salud del clima conversacional. Argumenta en contra de ajustar modelos para este problema. ¿Bastaría una regla, una búsqueda o un modelo preentrenado? ¿Qué tendría que demostrar nuestro proyecto para justificar el enfoque sin caer en censura injusta?"

# 2. Descripción y justificación del corpus

**Por qué este corpus (y no otro):** no buscamos un dataset "bonito" en inglés. Queremos comentarios reales en español, anotados por personas, donde el odio apunta a grupos concretos (mujeres, LGBTI, racismo, discapacidad, apariencia…). Eso conecta con la propuesta social: proteger a quien suele ser el blanco del ataque, no solo marcar "palabras feas".

| Campo | Tipo | Función |
|---|---|---|
| `id` | uint64 | Identificador del comentario |
| `title` | string | Contexto: título de la noticia |
| `text` | string | **Contenido principal** (comentario) |
| `context_tweet` | string | Contexto: tuit del medio con la noticia |
| `body` | string | Contexto: cuerpo de la noticia |
| `HATEFUL` | ClassLabel (Not Hateful / Hateful) | **Etiqueta objetivo** |
| `CALLS` | ClassLabel (NO/YES) | Anotación: llamado a la acción |
| `WOMEN`, `LGBTI`, `RACISM`, `CLASS`, `POLITICS`, `DISABLED`, `APPEARANCE`, `CRIMINAL` | ClassLabel (NO/YES) | Anotaciones por tipo de odio (quién es agredido) |

_Fuente: esquema consultado en la API de Hugging Face (nov 2026). Verificar con las celdas siguientes._

In [ ]:
!pip -q install datasets transformers accelerate evaluate scikit-learn pysentimiento

In [ ]:
from datasets import load_dataset
import pandas as pd, numpy as np

ds = load_dataset("piuba-bigdata/contextualized_hate_speech")
print(ds)
print(ds["train"].features)

# 3. Análisis exploratorio de datos (EDA)
Leemos el corpus con doble mirada: **técnica** (calidad, balance, longitudes, fugas) y **humana** (¿a quién apunta el odio? ¿hay agresiones sin groserías? ¿qué vería un menor si esto se publicara sin filtro?).

Cada subsección termina con una conclusión escrita por el equipo (**TODO**).

**Datos ya verificados** con la API de estadísticas de Hugging Face (nov 2026), antes de ejecutar este cuaderno:

| Partición | Filas | Hateful | % Hateful | Artículos únicos (`body`) | Comentarios por artículo |
|---|---|---|---|---|---|
| train | 36.420 | 5.531 | 15,2 % | 990 | ~36,8 |
| test | 11.343 | 1.797 | 15,8 % | 248 | ~45,7 |

Sin valores nulos en ninguna columna de etiqueta. Desbalance aproximado de 1 a 5,6. Las celdas siguientes deben reproducir estos números.

**Hipótesis de fuga:** el train ya contiene 990 artículos distintos, es decir, casi todos los del corpus. Es probable que los artículos del test también estén en el train. La celda 3.3 lo confirma o lo descarta.

In [ ]:
df = {k: ds[k].to_pandas() for k in ds}
tr, dv, te = df["train"], df["dev"], df["test"]

# 3.1 Inventario y calidad
for k, d in df.items():
    print(k, d.shape, "| vacíos en text:", d["text"].isna().sum() + (d["text"].str.strip() == "").sum(),
          "| duplicados exactos de text:", d["text"].duplicated().sum())

In [ ]:
# 3.2 Balance de clases por partición
for k, d in df.items():
    print(k, d["HATEFUL"].value_counts(normalize=True).round(3).to_dict())

cats = ["CALLS","WOMEN","LGBTI","RACISM","CLASS","POLITICS","DISABLED","APPEARANCE","CRIMINAL"]
print(tr.loc[tr.HATEFUL==1, cats].mean().sort_values(ascending=False).round(3))

In [ ]:
# 3.3 Fuga por artículo: ¿las mismas noticias aparecen en train y test?
key = "body"   # identifica el artículo; title tiene menos valores únicos que body
a_tr, a_dv, a_te = set(tr[key]), set(dv[key]), set(te[key])
print("artículos únicos train/dev/test:", len(a_tr), len(a_dv), len(a_te))
print("test con artículo visto en train:", te[key].isin(a_tr).mean().round(3))
print("dev con artículo visto en train:", dv[key].isin(a_tr).mean().round(3))

# ¿El odio depende del artículo? Si sí, el modelo puede aprender el tema en vez del comentario.
tasa = tr.groupby(key).HATEFUL.mean()
print("tasa de odio por artículo (train):", tasa.describe().round(3).to_dict())

**Decisión de partición** (resultado de 3.3, verificado):
- El **test oficial no comparte artículos (`body`) con train** (0 %). Sirve para comentarios nuevos sobre noticias nuevas.
- El **dev oficial está 100 % contaminado**: todos sus artículos ya aparecen en train. No mide generalización a noticias nuevas.
- Por eso la evaluación principal usa **partición por artículo** (`GroupShuffleSplit`, seed 42): train 36.587 / dev 8.971 / test 11.311 (~15 % hateful). Ningún artículo cruza particiones.
- Se reportan ambas lógicas cuando haga falta comparar con el paper original; **los cuatro modelos se miden en `split_articulo_*`.**


In [ ]:
# Partición por artículo: ningún artículo queda en dos particiones.
from sklearn.model_selection import GroupShuffleSplit
full = pd.concat([tr.assign(split_oficial="train"), dv.assign(split_oficial="dev"),
                  te.assign(split_oficial="test")], ignore_index=True)
groups = full[key].factorize()[0]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_trdv, idx_te = next(gss.split(full, groups=groups))
rest = full.iloc[idx_trdv]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
i_tr, i_dv = next(gss2.split(rest, groups=groups[idx_trdv]))

g_tr, g_dv, g_te = rest.iloc[i_tr], rest.iloc[i_dv], full.iloc[idx_te]
for n, d in [("train", g_tr), ("dev", g_dv), ("test", g_te)]:
    print(n, len(d), "| artículos:", d[key].nunique(), "| % hateful:", round(d.HATEFUL.mean()*100, 1))
assert not (set(g_tr[key]) & set(g_te[key])) and not (set(g_tr[key]) & set(g_dv[key]))

# Guardar para que todos los modelos usen exactamente las mismas filas.
cols = ["id", "text", "title", "context_tweet", "HATEFUL"] + cats
for n, d in [("train", g_tr), ("dev", g_dv), ("test", g_te)]:
    d[cols].to_parquet(f"split_articulo_{n}.parquet", index=False)
# TODO: copiar los .parquet a Google Drive compartido del equipo.

In [ ]:
# 3.4 Longitud en palabras y tokens (tokenizador de A3, con su preprocesamiento obligatorio)
tr["n_words"] = tr.text.str.split().str.len()
print(tr.n_words.describe().round(1))
print(tr.groupby("HATEFUL").n_words.mean().round(2))

from transformers import AutoTokenizer
from pysentimiento.preprocessing import preprocess_tweet
tok = AutoTokenizer.from_pretrained("pysentimiento/robertuito-base-uncased")   # verificado en el Hub (sha 20cb399)
muestra = [preprocess_tweet(t, lang="es") for t in tr.text.sample(2000, random_state=0)]
lens = [len(x) for x in tok(muestra)["input_ids"]]
print("tokens p50/p95/max:", np.percentile(lens, [50, 95, 100]), "| límite del modelo: 128")

In [ ]:
# 3.5 Muestras representativas, ambiguas y difíciles (leerlas, no solo mostrarlas)
pd.set_option("display.max_colwidth", 200)
print("HATEFUL=1"); display(tr[tr.HATEFUL==1].sample(8, random_state=1)[["text"]])
print("HATEFUL=0"); display(tr[tr.HATEFUL==0].sample(8, random_state=1)[["text"]])
# TODO: marcar casos con negación, sarcasmo, insulto sin odio y odio sin groserías.

In [ ]:
# 3.6 ¿Basta una lista de palabras? Regla ingenua evaluada en dev (evidencia para la Fase 0).
import re
from sklearn.metrics import precision_score, recall_score, f1_score
groserias = ["mierda", "puto", "puta", "boludo", "pelotudo", "idiota", "imbecil", "imbécil",
             "estupido", "estúpido", "basura", "forro", "negro de mierda", "kuka", "zurdo",
             "gorila", "chorro", "vago", "asco", "mamerto"]   # TODO: documentar origen y ampliar
patron = re.compile(r"\b(" + "|".join(map(re.escape, groserias)) + r")\b", re.IGNORECASE)
pred = dv.text.str.contains(patron).astype(int)
print("regla  P=%.3f R=%.3f F1=%.3f" % (precision_score(dv.HATEFUL, pred), recall_score(dv.HATEFUL, pred),
                                         f1_score(dv.HATEFUL, pred)))
print("odio SIN palabra de la lista:", ((dv.HATEFUL==1) & (pred==0)).sum(),
      "| palabra de la lista SIN odio:", ((dv.HATEFUL==0) & (pred==1)).sum())
display(dv[(dv.HATEFUL==1) & (pred==0)].sample(5, random_state=0)[["text"]])
display(dv[(dv.HATEFUL==0) & (pred==1)].sample(5, random_state=0)[["text"]])

**Conclusiones del EDA** (verificadas al ejecutar):
- Sin textos vacíos; hay duplicados exactos (train 1.206) — vigilar fugas por copia.
- Desbalance ~15 % hateful; categorías más frecuentes en odio: racismo, apariencia, criminales.
- **Fuga:** test oficial limpio respecto de train; **dev oficial no**.
- Longitud corta (mediana ~11 palabras); tokens RoBERTuito p95≈54 < 128.
- Regla de groserías en DEV: F1≈0,20 → no basta; justifica aprendizaje profundo.


# 4. Preparación y representación de los datos
Todas las decisiones se toman con **train** (lo que se aprende) y **dev** (lo que se elige). El test no se mira.

| Decisión | A1 MLP (TF-IDF) | A2 BiLSTM/GRU | A3 RoBERTuito |
|---|---|---|---|
| Partición | `split_articulo_*` (3.3) | igual | igual |
| Limpieza | `limpiar()`: URL y usuarios normalizados, minúsculas, puntuación separada | igual que A1 | `preprocess_tweet()` de pysentimiento (lo exige el modelo); no se limpia dos veces |
| Stopwords | se evalúa con y sin, **conservando negaciones** | no se eliminan | no se eliminan |
| Tokens | palabras y bigramas | vocabulario propio aprendido en train | tokenizador del modelo |
| Longitud | no aplica | p95 de train | 128 tokens (límite del modelo) |
| Desbalance | pesos de clase de train | igual | igual |

Mayúsculas y emojis pueden ser señal de agresión: la celda 4.2 mide si cambian entre clases antes de decidir borrarlos.

In [ ]:
# 4.1 Cargar la partición fija (la misma para los 4 modelos)
import re, unicodedata, json
SPLIT_DIR = "."   # en Colab: carpeta donde guardaron split_articulo_*.parquet
sp = {n: pd.read_parquet(f"{SPLIT_DIR}/split_articulo_{n}.parquet") for n in ["train", "dev", "test"]}
for n, d in sp.items():
    print(n, d.shape, "| % hateful:", round(d.HATEFUL.mean() * 100, 1))


In [ ]:
# 4.2 ¿Mayúsculas y emojis son señal? Medir antes de borrarlos.
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")
def prop_mayus(t):
    w = re.findall(r"[^\W\d_]{3,}", t)   # palabras de 3+ letras, sin puntuación pegada
    return np.mean([x.isupper() for x in w]) if w else 0.0

d = sp["train"]
print("prop. palabras en MAYÚSCULAS por clase:", d.text.map(prop_mayus).groupby(d.HATEFUL).mean().round(3).to_dict())
print("% con emoji por clase:", d.text.str.contains(EMOJI).groupby(d.HATEFUL).mean().round(3).to_dict())
print("% con URL:", round(d.text.str.contains(r"https?://").mean() * 100, 1),
      "| % con @usuario:", round(d.text.str.contains("@usuario").mean() * 100, 1))
# TODO conclusión: si las mayúsculas difieren entre clases, evaluar en dev una variable extra 'prop_mayus' para A1.

In [ ]:
# 4.3 Limpieza para A1 y A2 (A3 usa su propio preprocesamiento)
def limpiar(t, minus=True, quitar_tildes=False, separar_puntuacion=True):
    t = re.sub(r"https?://\S+", " URL ", t)
    t = re.sub(r"@\w+", " @usuario ", t)
    if minus:
        t = t.lower()
    if quitar_tildes:
        t = "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")
    if separar_puntuacion:
        t = re.sub(r"([!?¡¿.,;:()\"])", r" \1 ", t)
    return re.sub(r"\s+", " ", t).strip()

def prep_a3(t):
    return preprocess_tweet(t, lang="es")

# Antes / después sobre casos que nos importan: negación, mayúsculas, emojis, URL
casos = pd.concat([
    d[d.text.str.contains(r"\b(no|nunca|ni)\b", case=False)].sample(3, random_state=0),
    d[d.text.map(prop_mayus) > 0.5].sample(2, random_state=0),
    d[d.text.str.contains(EMOJI)].sample(2, random_state=0),
    d[d.text.str.contains(r"https?://")].sample(1, random_state=0),
])
pd.set_option("display.max_colwidth", 160)
display(pd.DataFrame({"original": casos.text, "A1/A2: limpiar()": casos.text.map(limpiar),
                      "A3: preprocess_tweet()": casos.text.map(prep_a3), "HATEFUL": casos.HATEFUL}))
# TODO conclusión: ¿alguna transformación borra una negación o cambia el sentido?

In [ ]:
# 4.4 Stopwords: la lista estándar de español incluye negaciones. Comprobarlo.
import nltk; nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
sw = set(stopwords.words("spanish"))
negaciones = ["no", "ni", "nunca", "jamás", "sin", "nada", "nadie", "tampoco"]
print("negaciones dentro de la lista NLTK:", [w for w in negaciones if w in sw])
sw_sin_neg = sorted(sw - set(negaciones))   # lista a usar en el experimento de A1

In [ ]:
# 4.5 A3: tokenización y truncamiento (solo comentario y comentario + título de la noticia)
tok3 = AutoTokenizer.from_pretrained("pysentimiento/robertuito-base-uncased")
print("¿el tokenizador pasa a minúsculas solo?", tok3.tokenize("HOLA MUNDO") == tok3.tokenize("hola mundo"))
# TODO: si imprime False, añadir .lower() en prep_a3 (el modelo es uncased) y volver a correr.

MAX_LEN = 128
txt = [prep_a3(t) for t in sp["train"].text]
ctx = [prep_a3(t) for t in sp["train"].title]
l_txt = np.array([len(x) for x in tok3(txt)["input_ids"]])
l_par = np.array([len(x) for x in tok3(txt, ctx)["input_ids"]])
print("solo comentario  p95=%d  truncados=%.2f%%" % (np.percentile(l_txt, 95), (l_txt > MAX_LEN).mean() * 100))
print("comentario+título p95=%d truncados=%.2f%%" % (np.percentile(l_par, 95), (l_par > MAX_LEN).mean() * 100))
# Para H3 se usa truncation="only_second": se recorta el título, nunca el comentario.

In [ ]:
# 4.6 A1: TF-IDF ajustado SOLO con train
from sklearn.feature_extraction.text import TfidfVectorizer
def hacer_tfidf(stop=None):
    return TfidfVectorizer(preprocessor=limpiar, token_pattern=r"(?u)\b\w+\b|[!?¡¿]",
                           ngram_range=(1, 2), min_df=3, max_features=50_000,
                           sublinear_tf=True, stop_words=stop)
tfidf = hacer_tfidf()
X1 = {"train": tfidf.fit_transform(sp["train"].text)}
X1.update({n: tfidf.transform(sp[n].text) for n in ["dev", "test"]})
print({n: X.shape for n, X in X1.items()})
# Experimento (dev): hacer_tfidf(sw_sin_neg) vs hacer_tfidf(None). Registrar el resultado en 5.2.

In [ ]:
# 4.7 A2: vocabulario propio aprendido SOLO con train
import tensorflow as tf
limpio = {n: sp[n].text.map(limpiar).values for n in sp}
n_tok = np.array([len(t.split()) for t in limpio["train"]])
SEQ_LEN = int(np.percentile(n_tok, 95))
vec = tf.keras.layers.TextVectorization(max_tokens=30_000, output_sequence_length=SEQ_LEN,
                                        standardize=None, split="whitespace")
vec.adapt(limpio["train"])
X2 = {n: vec(limpio[n]).numpy() for n in sp}

vocab = set(vec.get_vocabulary())
oov = np.mean([w not in vocab for t in limpio["dev"] for w in t.split()])
print("SEQ_LEN:", SEQ_LEN, "| vocabulario:", len(vocab), "| % palabras de dev fuera del vocabulario: %.1f" % (oov * 100))
print("% comentarios de train truncados:", round((n_tok > SEQ_LEN).mean() * 100, 1))

In [ ]:
# 4.8 Desbalance y configuración guardada (la app debe leer este mismo archivo)
from sklearn.utils.class_weight import compute_class_weight
y = {n: sp[n].HATEFUL.values for n in sp}
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y["train"])
print("pesos de clase (train):", pesos.round(3))

prep_config = {"split": "split_articulo", "a3_model": "pysentimiento/robertuito-base-uncased",
               "a3_max_len": MAX_LEN, "a3_preprocess": "pysentimiento.preprocess_tweet(lang='es')",
               "a2_seq_len": SEQ_LEN, "limpiar": {"minus": True, "quitar_tildes": False, "separar_puntuacion": True},
               "class_weights": pesos.round(4).tolist()}
json.dump(prep_config, open("prep_config.json", "w"), indent=2, ensure_ascii=False)
print(prep_config)

**Conclusiones de la preparación** (evidencia de las celdas 4.2–4.8):
- **Mayúsculas y emojis (4.2):** diferencia pequeña entre clases; no justifican un preprocesado agresivo. `@usuario` aparece en ~100 % del texto (ya anonimizado en el corpus).
- **Transformaciones (4.3):** `limpiar()` separa puntuación y pasa a minúsculas sin borrar negaciones. A3 usa solo `preprocess_tweet` (no se limpia dos veces).
- **Stopwords (4.4):** NLTK incluye `no`, `ni`, `sin`, `nada`. En A1 se probó con/sin stopwords **conservando negaciones**; la mejor config fue **sin stopwords**.
- **Truncamiento A3 (4.5):** el tokenizador **sí** hace lowercasing. Con máx. 128, el truncamiento es ~0 % (comentario solo o + título).
- **A2 (4.7):** `SEQ_LEN`≈47 (p95), vocabulario 30k, OOV en dev ≈4 %.
- **Desbalance (4.8):** pesos de clase train ≈ `[0.59, 3.25]`.


# 5. Modelado y evaluación

Semilla fija **42**. Misma partición `split_articulo_*` para A1, A2, A3 y la línea base L. El **test no se usa** para elegir hiperparámetros ni umbral.

## 5.1 Arquitecturas
| # | Arquitectura | Representación | Preentrenamiento | Qué se entrena |
|---|---|---|---|---|
| A1 | MLP (capas densas) | TF-IDF palabras + bigramas | Ninguno | Capas densas (`sklearn.MLPClassifier`) |
| A2 | BiLSTM | Índices de vocabulario propio + embedding | Ninguno | Embedding + BiLSTM + cabeza |
| A3 | Transformer RoBERTuito | `preprocess_tweet` + tokens, máx. 128 | `pysentimiento/robertuito-base-uncased` | Fine-tuning + cabeza |
| L | Línea base sin ajuste | igual que A3 | `pysentimiento/robertuito-hate-speech` (multietiqueta) | Nada (solo inferencia) |

**Solapamiento a declarar:** RoBERTuito y el corpus vienen del grupo UBA (tuits). Posible solapamiento con el preentrenamiento (arXiv 2111.09453).

## 5.2 Métricas
Accuracy, Precision, Recall, F1 (clase *Hateful*), matriz de confusión, ROC-AUC. Umbral elegido en **DEV** maximizando F1 de la clase positiva (prioriza no dejar pasar odio, con el costo de más falsos positivos).

### Resultados verificados en TEST (seed 42)
| Modelo | F1 odio | Precision | Recall | ROC-AUC | Umbral (dev) | Tiempo ent. (s) | Latencia (ms) |
|---|---:|---:|---:|---:|---:|---:|---:|
| A1 MLP + TF-IDF | 0.495 | 0.533 | 0.462 | 0.796 | 0.761 | 230 | 0.04 |
| A2 BiLSTM | 0.544 | 0.639 | 0.473 | 0.819 | 0.751 | 129 | 2.3 |
| **A3 RoBERTuito** | **0.653** | **0.686** | **0.623** | **0.898** | **0.280** | 3283 | 15.4 |
| L robertuito-hate-speech | 0.376 | 0.358 | 0.396 | 0.678 | 0.473 | 0 | 16.2 |

**Selección final: A3.** Mejor F1 y AUC. L es multietiqueta (`hateful`/`targeted`/`aggressive`) con otra definición; queda como línea base honesta, no como competidor idéntico.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_recall_curve
import numpy as np

def best_threshold(y_true, y_prob):
    p, r, t = precision_recall_curve(y_true, y_prob)
    f1 = (2 * p * r) / np.clip(p + r, 1e-12, None)
    i = int(np.nanargmax(f1[:-1])) if len(t) else 0
    return float(t[i]) if len(t) else 0.5

def evaluate(y_true, y_prob, thr=None, nombre="modelo"):
    if thr is None:
        thr = best_threshold(y_true, y_prob)
    y_pred = (y_prob >= thr).astype(int)
    print("===", nombre, "| thr=", round(thr, 3), "===")
    print(classification_report(y_true, y_pred, digits=3))
    print("cm", confusion_matrix(y_true, y_pred).tolist(),
          "| ROC-AUC", round(roc_auc_score(y_true, y_prob), 3),
          "| F1_hateful", round(f1_score(y_true, y_pred, pos_label=1), 3))
    return thr


In [ ]:
# --- A1: MLP + TF-IDF (búsqueda acotada en DEV) ---
# Requiere sp, limpiar, hacer_tfidf de la sección 4.
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
import time

tfidf = hacer_tfidf(stop=None)
X_tr = tfidf.fit_transform(sp["train"].text)
X_dv = tfidf.transform(sp["dev"].text)
X_te = tfidf.transform(sp["test"].text)
y_tr, y_dv, y_te = sp["train"].HATEFUL.values, sp["dev"].HATEFUL.values, sp["test"].HATEFUL.values
sw = compute_sample_weight("balanced", y_tr)

clf1 = MLPClassifier(hidden_layer_sizes=(256,), alpha=1e-3, batch_size=256, max_iter=30,
                     early_stopping=True, random_state=42)
t0 = time.time()
clf1.fit(X_tr, y_tr, sample_weight=sw)
print("train s", round(time.time()-t0, 1))
prob_dv = clf1.predict_proba(X_dv)[:, 1]
thr1 = evaluate(y_dv, prob_dv, nombre="A1 DEV")
evaluate(y_te, clf1.predict_proba(X_te)[:, 1], thr=thr1, nombre="A1 TEST")


In [ ]:
# --- A2: BiLSTM (TensorFlow/Keras) ---
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

limpio = {n: sp[n].text.map(limpiar).values for n in sp}
n_tok = np.array([len(t.split()) for t in limpio["train"]])
SEQ_LEN = int(np.percentile(n_tok, 95))
vec = tf.keras.layers.TextVectorization(max_tokens=30_000, output_sequence_length=SEQ_LEN,
                                        standardize=None, split="whitespace")
vec.adapt(limpio["train"])
X2 = {n: vec(limpio[n]).numpy() for n in sp}
y2 = {n: sp[n].HATEFUL.values.astype("float32") for n in sp}
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y2["train"].astype(int))
cw = {0: float(pesos[0]), 1: float(pesos[1])}

inp = tf.keras.Input(shape=(SEQ_LEN,), dtype="int32")
x = tf.keras.layers.Embedding(len(vec.get_vocabulary()), 100, mask_zero=True)(inp)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64))(x)
x = tf.keras.layers.Dropout(0.4)(x)
out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
m2 = tf.keras.Model(inp, out)
m2.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["AUC"])
m2.fit(X2["train"], y2["train"], validation_data=(X2["dev"], y2["dev"]),
       epochs=8, batch_size=128, class_weight=cw,
       callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_AUC", mode="max", patience=2, restore_best_weights=True)],
       verbose=2)
prob_dv = m2.predict(X2["dev"], batch_size=256).ravel()
thr2 = evaluate(y2["dev"], prob_dv, nombre="A2 DEV")
evaluate(y2["test"], m2.predict(X2["test"], batch_size=256).ravel(), thr=thr2, nombre="A2 TEST")


In [ ]:
# --- A3: fine-tuning RoBERTuito (GPU T4 recomendada en Colab) ---
# Puede tardar ~30–60+ min. Resultados reportados: F1_test≈0.653, thr≈0.28, seed=42.
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, set_seed
from datasets import Dataset
import torch, time

set_seed(42)
MODEL_A3 = "pysentimiento/robertuito-base-uncased"
tok3 = AutoTokenizer.from_pretrained(MODEL_A3)

def to_hf(df):
    texts = [preprocess_tweet(t, lang="es") for t in df.text.tolist()]
    enc = tok3(texts, truncation=True, padding="max_length", max_length=128)
    enc["labels"] = df.HATEFUL.astype(int).tolist()
    return Dataset.from_dict(enc)

ds_tr, ds_dv, ds_te = to_hf(sp["train"]), to_hf(sp["dev"]), to_hf(sp["test"])
model3 = AutoModelForSequenceClassification.from_pretrained(MODEL_A3, num_labels=2)
args = TrainingArguments(
    output_dir="a3_run", num_train_epochs=3, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, learning_rate=2e-5, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False, seed=42, report_to=[],
)
trainer = Trainer(model=model3, args=args, train_dataset=ds_tr, eval_dataset=ds_dv)
t0 = time.time(); trainer.train(); print("train s", round(time.time()-t0, 1))

def probs_from_trainer(ds):
    logits = trainer.predict(ds).predictions
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return (e[:, 1] / e.sum(axis=1))

thr3 = evaluate(np.array(ds_dv["labels"]), probs_from_trainer(ds_dv), nombre="A3 DEV")
evaluate(np.array(ds_te["labels"]), probs_from_trainer(ds_te), thr=thr3, nombre="A3 TEST")
trainer.save_model("a3_model"); tok3.save_pretrained("a3_model")


## 5.3 Línea base sin ajuste
`pysentimiento/robertuito-hate-speech`: `id2label = {0: hateful, 1: targeted, 2: aggressive}` (**multietiqueta**). Usamos el score de `hateful` y un umbral elegido en DEV.

**Resultado TEST:** F1=0.376, P=0.358, R=0.396, AUC=0.678. Queda por debajo de A1/A2/A3: adaptar al corpus aporta.


In [ ]:
# Línea base L (sin ajuste) sobre el mismo test por artículo.
# En Colab puede tardar varios minutos.
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
from pysentimiento.preprocessing import preprocess_tweet

MODEL_L = "pysentimiento/robertuito-hate-speech"
tok_l = AutoTokenizer.from_pretrained(MODEL_L)
mod_l = AutoModelForSequenceClassification.from_pretrained(MODEL_L)
print("id2label:", mod_l.config.id2label)
clf_l = pipeline("text-classification", model=mod_l, tokenizer=tok_l, truncation=True, max_length=128, top_k=None)

def hate_prob(out):
    scores = {d["label"].lower(): float(d["score"]) for d in out}
    return scores.get("hateful", max(scores.values()))

# DEV → umbral; TEST → una vez
texts_dv = [preprocess_tweet(t, lang="es") for t in sp["dev"].text.tolist()]
prob_dv = np.array([hate_prob(o) for o in clf_l(texts_dv, batch_size=32)])
thr_l = evaluate(sp["dev"].HATEFUL.values, prob_dv, nombre="L DEV")
texts_te = [preprocess_tweet(t, lang="es") for t in sp["test"].text.tolist()]
prob_te = np.array([hate_prob(o) for o in clf_l(texts_te, batch_size=32)])
evaluate(sp["test"].HATEFUL.values, prob_te, thr=thr_l, nombre="L TEST")


## 5.4 Tabla comparativa (calidad, tiempo y costo)
Ver tabla de la sección 5.2. **Decisión:** A3 por calidad. Para 100 000 comentarios/mes, la latencia de A3 (~15 ms) implica ≈0,4 h de cómputo de inferencia local; sin costo por request (a diferencia de una API LLM). Inversión: ~55 min de fine-tuning en MPS (o Colab T4).


## 5.5 Análisis de errores e iteración

**HateCheck-es** (n=3745, umbral DEV de A3): F1 global 0.652 (P=0.833, R=0.535).
- Débil en: ortografía alterada, odio implícito, **negaciones**, **contra-discurso** (muchos falsos positivos).
- Implicación empática: el botón “mostrar de todos modos” mitiga el daño de censurar a quien denuncia el odio citándolo.

**Fuera de dominio (ALIA, YouTube/TikTok, etiquetas por mayoría de 3 LLMs, n=4000):** F1≈0.28, recall≈0.17. El modelo de tuits argentinos no generaliza bien: límite a declarar.

### Decisión de mejora (comprobada con test fijo)
Comparar umbral **0.5** vs umbral elegido en DEV (**≈0.28**):
- TEST F1: 0.633 → **0.653** (más recall).
- Costo: más FP en casos sensibles de HateCheck (82 → 135).
- **Decisión:** quedarnos con el umbral DEV para cuidar a quien lee; documentar el trade-off.


In [ ]:
# HateCheck-spanish con el mejor modelo (A3). Requiere el modelo fine-tuned cargado.
# Si aún no se entrenó A3 en esta sesión, cargar desde la carpeta guardada o re-ejecutar 5.x.
from datasets import load_dataset
import pandas as pd

hc = load_dataset("Paul/hatecheck-spanish", split="test").to_pandas()
y_hc = (hc.label_gold.str.lower() == "hateful").astype(int).values
# TODO_COLAB: reutilizar la función de inferencia de A3 (preprocess_tweet + softmax clase 1)
# Placeholder: muestra distribución de funcionalidades a inspeccionar tras obtener predicciones
print(hc.functionality.value_counts().head(15))
print("casos:", len(hc), "| % hateful gold:", round(y_hc.mean()*100,1))
print("Tras predecir con A3, agrupar accuracy/F1 por functionality (ver resultados ya reportados arriba).")


## 5.6 Recursos de cómputo y costos
- **Máquina de los experimentos reportados:** Apple Silicon (MPS) para A3 y L; CPU/TensorFlow para A2; sklearn CPU para A1.
- **A3:** ~3283 s de entrenamiento (~55 min), early stopping ~2 épocas, batch 16, lr 2e-5, seed 42.
- **Volumen supuesto:** 100 000 comentarios/mes → A3 ≈ 0,43 h de inferencia; A1 casi gratis en latencia pero peor calidad.
- **Conclusión económica:** conviene A3 local (calidad + privacidad + sin tarifa por request). Un LLM por API solo sería atractivo si no se puede hospedar el modelo; hay que cotizar proveedor y fecha reales si se presenta esa opción.


In [ ]:
import platform, torch, psutil
print(platform.platform(), "| RAM GB:", round(psutil.virtual_memory().total/1e9,1))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no")

# 6. Aplicación web
La app es el gesto concreto del pitch: **alguien publica, alguien más podría leerlo; el sistema interviene con cuidado.**

1. **App del curso:** Gradio en Hugging Face Spaces con el modelo **A3**, umbral ≈0.28, aviso de límites (español, longitud, no sustituye humanos).
2. **Consistencia:** mismos ejemplos en cuaderno y app (registrar 5 al desplegar).
3. **Demo del pitch:** blog estático (`blog/index.html`). Si la probabilidad supera el umbral, el comentario se reemplaza por un aviso de normas de la comunidad, con “mostrar de todos modos”.
4. **Extensión de navegador:** la hace el equipo; reutiliza la API del Space.

Enlace Space: _(completar al desplegar en Hugging Face)_  
Blog: subir `blog/index.html` a un Space estático o servirlo junto al Gradio.


# 7. Limitaciones, condiciones de uso y mejoras futuras
- **No es terapia ni prevención clínica:** solo reduce exposición a odio.
- **No es censura absoluta:** umbral + aviso + opción de mostrar / revisar.
- **Dominio:** español rioplatense / pandemia; en ALIA (YouTube/TikTok) el recall cae fuerte.
- **Definiciones:** odio (piuba) ≠ multietiqueta de la línea base.
- **Contra-discurso y negaciones:** falsos positivos/negativos frecuentes (HateCheck).
- **Privacidad:** el texto sale del navegador hacia la API del Space.
- **Contenido sensible** en el corpus: cuidado al mostrar muestras en la presentación.
- **Mejoras:** más dialectos, umbral por tipo de comunidad, modelo on-device en la extensión, reentrenar con datos de comentarios (no solo tuits).


# 8. Reparto de trabajo
| Persona | Responsabilidad |
|---|---|
| _(TODO)_ | EDA y preparación |
| _(TODO)_ | Modelos |
| _(TODO)_ | Línea base y costos |
| _(TODO)_ | App web, blog, extensión y presentación |

# 9. Registro de revisión con IA
Cada observación de la IA se verificó con celdas o experimentos antes de usarla como hallazgo.

| Etapa | Prompt (resumen) | Evidencia | Crítica (síntesis) | Comprobación | Decisión |
|---|---|---|---|---|---|
| Fase 0 | Cuidador de adolescente: ¿por qué no confiar en ocultar odio con aviso? | Pitch + HATEFUL + corpus | Censura injusta; no medir salud mental; sarcasmo; privacidad | HateCheck FP en contra-discurso; límites declarados | Filtro previo + humano; no prometer “salud mental” |
| EDA | ¿Qué implica la fuga por artículo? | test∩train=0 %; dev∩train=100 %; F1 regla 0,20 | DEV oficial no mide noticias nuevas | Reproducido en 3.3 | Evaluación = `split_articulo_*` |
| Preparación | ¿La limpieza borra señales? | 4.2–4.5 | Conservar negaciones; no limpiar dos veces en A3 | Stopwords NLTK incluyen negaciones; tokenizador uncased | A1 sin stopwords; A3 solo `preprocess_tweet` |
| Modelado | ¿Umbral por recall o precisión? | Tabla A1–A3–L; 0,5 vs 0,28 | Umbral bajo ↑FP en contra-discurso | F1 test 0,633→0,653; FP sensibles 82→135 | A3 + umbral DEV; documentar trade-off |
| App | Cómo comunicar límites sin “seguridad total” | Blog + Gradio | Probar vacío, otro idioma, contra-discurso | Smoke: limpio≈0,03 / agresivo≈0,995 | Avisos claros + “mostrar de todos modos” |

La IA solo vio texto (no tono ni contexto social completo).
